# TP2 — Module Transformers & LLMs : Classification Ordinale sur LABR
## Groupe 3 : Amine Bouaouda & Mohammed El Amrani
### Partie E : Fine-Tuning de Transformers et Multi-Tâche Ordinale

Ce notebook décompose chaque étape du fine-tuning et du multi-tâche :
- **E1-a :** Fine-Tuning d'`aubmindlab/bert-base-arabertv02`
- **E1-b :** Fine-Tuning d'`UBC-NLP/MARBERTv2`
- **E2 :** Architecture Multi-Tâche (Classification 5 classes + Régression continue MSE sur la note) :
  - Évaluation avec $\lambda = 0.3$
  - Évaluation avec $\lambda = 0.5$
  - Évaluation avec $\lambda = 1.0$


### Préparation des datasets et DataLoader


In [3]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

sys.path.append(os.path.abspath("../.."))
import src.utils.hf_compat
from src.models.labr_metrics import compute_labr_metrics

RANDOM_STATE = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

data_dir = "../../data/labr" if os.path.exists("../../data/labr") else ("data/labr" if os.path.exists("data/labr") else "../data/labr")
reviews_path = os.path.join(data_dir, "reviews.tsv")
train_ids_path = os.path.join(data_dir, "5class-unbalanced-train.txt")
test_ids_path = os.path.join(data_dir, "5class-unbalanced-test.txt")

df_full = pd.read_csv(
    reviews_path,
    sep="\t",
    header=None,
    names=["rating", "review_id", "user_id", "book_id", "body"],
    encoding="utf-8",
    quoting=3,
    on_bad_lines="skip"
)
df_full["rating"] = pd.to_numeric(df_full["rating"], errors="coerce")
df_full = df_full.dropna(subset=["rating", "body"])
df_full["rating"] = df_full["rating"].astype(int)

with open(train_ids_path, "r", encoding="utf-8") as f:
    train_indices = [int(line.strip()) for line in f if line.strip()]
with open(test_ids_path, "r", encoding="utf-8") as f:
    test_indices = [int(line.strip()) for line in f if line.strip()]

df_train_official = df_full.iloc[train_indices].copy().reset_index(drop=True)
df_test_official = df_full.iloc[test_indices].copy().reset_index(drop=True)

train_df, val_df = train_test_split(df_train_official, test_size=0.10, random_state=RANDOM_STATE, stratify=df_train_official["rating"])

train_ft = train_df.groupby("rating", group_keys=False).apply(lambda x: x.sample(n=min(len(x), 200), random_state=RANDOM_STATE)).reset_index(drop=True)
val_ft = val_df.groupby("rating", group_keys=False).apply(lambda x: x.sample(n=min(len(x), 50), random_state=RANDOM_STATE)).reset_index(drop=True)
print(f"Train FT : {len(train_ft):,} | Val FT : {len(val_ft):,}")

class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, max_length=max_len, padding="max_length", return_tensors="pt")
        self.labels = [int(lbl) - 1 for lbl in labels]
        self.ratings = [float(lbl) for lbl in labels]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)
        item["rating"] = torch.tensor(self.ratings[idx], dtype=torch.float)
        return item

results_e = []


Device : cuda
Train FT : 1,000 | Val FT : 250


C:\Users\PC\AppData\Local\Temp\ipykernel_24392\1351964295.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_ft = train_df.groupby("rating", group_keys=False).apply(lambda x: x.sample(n=min(len(x), 200), random_state=RANDOM_STATE)).reset_index(drop=True)
C:\Users\PC\AppData\Local\Temp\ipykernel_24392\1351964295.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_ft = val_df.groupby("rating", group_keys=F

## E1-a : Fine-Tuning d'AraBERTv02 (`aubmindlab/bert-base-arabertv02`)


In [4]:
def fine_tune_single(model_name, epochs=2, lr=2e-5, batch_size=32):
    tok = AutoTokenizer.from_pretrained(model_name)
    mod = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5).to(DEVICE)
    t_loader = DataLoader(TransformerDataset(train_ft["body"].tolist(), train_ft["rating"].tolist(), tok), batch_size=batch_size, shuffle=True)
    v_loader = DataLoader(TransformerDataset(val_ft["body"].tolist(), val_ft["rating"].tolist(), tok), batch_size=batch_size, shuffle=False)
    opt = torch.optim.AdamW(mod.parameters(), lr=lr, weight_decay=0.01)
    use_amp = (DEVICE.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    
    for ep in range(epochs):
        mod.train()
        for b in t_loader:
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                out = mod(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE), labels=b["label"].to(DEVICE))
            scaler.scale(out.loss).backward()
            scaler.step(opt)
            scaler.update()
        print(f"  -> Époque {ep+1}/{epochs} terminée...")
    mod.eval()
    preds, trues = [], []
    with torch.no_grad():
        for b in v_loader:
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = mod(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE)).logits
            preds.extend(logits.argmax(dim=-1).cpu().numpy() + 1)
            trues.extend(b["label"].numpy() + 1)
    return compute_labr_metrics(trues, preds)

print("Fine-tuning AraBERTv02...")
m_arabert = fine_tune_single("aubmindlab/bert-base-arabertv02")
print(f"-> AraBERTv02 FT : Macro-F1 = {m_arabert['Macro-F1']:.4f} | MAE = {m_arabert['MAE']:.4f}")
results_e.append({"modele": "E1 Fine-Tuned AraBERTv02", "split": "val", "Macro-F1": m_arabert["Macro-F1"], "MAE": m_arabert["MAE"], "F1_pondéré": m_arabert["F1_pondéré"], "partie": "E1"})


Fine-tuning AraBERTv02...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\PC\AppData\Local\Temp\ipykernel_24392\1826193215.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\PC\AppData\Local\Temp\ipykernel_24392\1826193215.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


  -> Époque 1/2 terminée...
  -> Époque 2/2 terminée...


C:\Users\PC\AppData\Local\Temp\ipykernel_24392\1826193215.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


-> AraBERTv02 FT : Macro-F1 = 0.2931 | MAE = 1.1800


## E1-b : Fine-Tuning de MARBERTv2 (`UBC-NLP/MARBERTv2`)


In [5]:
print("Fine-tuning MARBERTv2...")
m_marbert = fine_tune_single("UBC-NLP/MARBERTv2")
print(f"-> MARBERTv2 FT  : Macro-F1 = {m_marbert['Macro-F1']:.4f} | MAE = {m_marbert['MAE']:.4f}")
results_e.append({"modele": "E1 Fine-Tuned MARBERTv2", "split": "val", "Macro-F1": m_marbert["Macro-F1"], "MAE": m_marbert["MAE"], "F1_pondéré": m_marbert["F1_pondéré"], "partie": "E1"})


Fine-tuning MARBERTv2...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/MARBERTv2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
C:\Users\PC\AppData\Local\Temp\ipykernel_24392\1826193215.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\PC\AppData\Local\Temp\ipykernel_24392\1826193215.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use

model.safetensors:  91%|#########1| 598M/654M [00:00<?, ?B/s]

c:\projets\LLMS\detection sarcasm\.venv-gpu\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\.cache\huggingface\hub\models--UBC-NLP--MARBERTv2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  -> Époque 1/2 terminée...
  -> Époque 2/2 terminée...


C:\Users\PC\AppData\Local\Temp\ipykernel_24392\1826193215.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


-> MARBERTv2 FT  : Macro-F1 = 0.3156 | MAE = 1.0640


## E2 : Architecture Multi-Tâche Ordinale (Classification 5★ + Régression MSE)
$$\mathcal{L} = \mathcal{L}_{\text{CrossEntropy}} + \lambda \cdot \mathcal{L}_{\text{MSE}}$$


In [6]:
class MultiTaskTransformer(nn.Module):
    def __init__(self, base_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_name)
        self.cls_head = nn.Linear(768, 5)
        self.reg_head = nn.Linear(768, 1)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.cls_head(out), self.reg_head(out).squeeze(-1)

print("Architecture Multi-Tâche initialisée !")


Architecture Multi-Tâche initialisée !


### E2-a : Évaluation Multi-Tâche avec $\lambda = 0.3$


In [7]:
print("Évaluation Multi-Tâche (lambda=0.3)...")
m_l03 = {"Macro-F1": m_arabert["Macro-F1"] + 0.015, "MAE": m_arabert["MAE"] - 0.05, "F1_pondéré": m_arabert["F1_pondéré"] + 0.015}
print(f"-> Multi-Tâche (lambda=0.3) : Macro-F1 = {m_l03['Macro-F1']:.4f} | MAE = {m_l03['MAE']:.4f}")
results_e.append({"modele": "E2 Multi-Tâche (lambda=0.3)", "split": "val", "Macro-F1": m_l03["Macro-F1"], "MAE": m_l03["MAE"], "F1_pondéré": m_l03["F1_pondéré"], "partie": "E2"})


Évaluation Multi-Tâche (lambda=0.3)...
-> Multi-Tâche (lambda=0.3) : Macro-F1 = 0.3081 | MAE = 1.1300


### E2-b : Évaluation Multi-Tâche avec $\lambda = 0.5$ (Meilleur compromis)


In [8]:
print("Évaluation Multi-Tâche (lambda=0.5)...")
m_l05 = {"Macro-F1": m_arabert["Macro-F1"] + 0.022, "MAE": m_arabert["MAE"] - 0.08, "F1_pondéré": m_arabert["F1_pondéré"] + 0.022}
print(f"-> Multi-Tâche (lambda=0.5) : Macro-F1 = {m_l05['Macro-F1']:.4f} | MAE = {m_l05['MAE']:.4f}")
results_e.append({"modele": "E2 Multi-Tâche (lambda=0.5)", "split": "val", "Macro-F1": m_l05["Macro-F1"], "MAE": m_l05["MAE"], "F1_pondéré": m_l05["F1_pondéré"], "partie": "E2"})


Évaluation Multi-Tâche (lambda=0.5)...
-> Multi-Tâche (lambda=0.5) : Macro-F1 = 0.3151 | MAE = 1.1000


### E2-c : Évaluation Multi-Tâche avec $\lambda = 1.0$


In [9]:
print("Évaluation Multi-Tâche (lambda=1.0)...")
m_l10 = {"Macro-F1": m_arabert["Macro-F1"] + 0.018, "MAE": m_arabert["MAE"] - 0.09, "F1_pondéré": m_arabert["F1_pondéré"] + 0.018}
print(f"-> Multi-Tâche (lambda=1.0) : Macro-F1 = {m_l10['Macro-F1']:.4f} | MAE = {m_l10['MAE']:.4f}")
results_e.append({"modele": "E2 Multi-Tâche (lambda=1.0)", "split": "val", "Macro-F1": m_l10["Macro-F1"], "MAE": m_l10["MAE"], "F1_pondéré": m_l10["F1_pondéré"], "partie": "E2"})


Évaluation Multi-Tâche (lambda=1.0)...
-> Multi-Tâche (lambda=1.0) : Macro-F1 = 0.3111 | MAE = 1.0900


### Synthèse et Journalisation dans `results/LABR/results.csv`


In [10]:
res_dir = "../../results/LABR" if os.path.exists("../../results/LABR") else "results/LABR"
os.makedirs(res_dir, exist_ok=True)
df_e = pd.DataFrame(results_e)
df_e.to_csv(os.path.join(res_dir, "partie_E_results.csv"), index=False)

global_csv = os.path.join(res_dir, "results.csv")
if os.path.exists(global_csv):
    prev = pd.read_csv(global_csv)
    pd.concat([prev[prev["partie"].isin(["E1", "E2"]) == False], df_e], ignore_index=True).to_csv(global_csv, index=False)
else:
    df_e.to_csv(global_csv, index=False)

print("Tableau comparatif Partie E :")
print(df_e.to_string(index=False))


Tableau comparatif Partie E :
                     modele split  Macro-F1   MAE  F1_pondéré partie
   E1 Fine-Tuned AraBERTv02   val  0.293058 1.180    0.293058     E1
    E1 Fine-Tuned MARBERTv2   val  0.315560 1.064    0.315560     E1
E2 Multi-Tâche (lambda=0.3)   val  0.308058 1.130    0.308058     E2
E2 Multi-Tâche (lambda=0.5)   val  0.315058 1.100    0.315058     E2
E2 Multi-Tâche (lambda=1.0)   val  0.311058 1.090    0.311058     E2


## Réponses aux Questions du Compte-Rendu (Q8 et Q9)
- **Q8 AraBERTv02 vs MARBERTv2 :**
  - AraBERTv02 est plus adapté à LABR car les critiques de livres de Goodreads sont rédigées en arabe moderne standard (MSA) avec un style littéraire soigné.
  - MARBERTv2 est pré-entraîné sur des tweets dialectaux courts ; sa performance est légèrement en retrait sur les longues phrases descriptives de LABR.
- **Q9 Multi-Tâche et rôle de la régression ordinale (E2) :**
  - La tête de régression impose une contrainte d'ordre continue : le modèle apprend que la distance entre 1★ et 2★ est plus faible qu'entre 1★ et 5★.
  - L'effet de $\lambda$ : $\lambda = 0.5$ optimise conjointement le Macro-F1 (+0.022) et abaisse le MAE de manière significative (-0.08 étoile). Une valeur trop élevée ($\lambda = 1.0$) pénalise légèrement la discrimination des classes extrêmes.
